# Signpost-D one-pass on all the data (v1_3), with two switches for score questions

Runs 11 and 12 (`08b`, `08c`) showed that **choice accuracy follows the amount of data, and training on all three types together gives much better probabilities** than training on one type. So this notebook trains run 7's encoder on **every question of v1_3**: v1_1 plus the 150 choice cases (K) and the 299 score cases (L), about 8,000 training questions. Run 7's settings otherwise: start `skamalj/signpost-one-pass`, LoRA r 64, `HEAD_EXTRA` 2, seed 42, cross-entropy; **3 epochs** (the best epoch has been 2 in every run from this start).

**Dev:** v1, the invoices (I), the choice cases (K) and the score cases (L) are each split on their own, so every earlier dev case stays a dev case and 15% of the L cases join.

**Two switches for score questions** (first code cell, both off by default; the question's documented `type` picks the head, on our data and on the benchmark):
- **`ORDINAL_HEAD`:** a score question's probabilities form one peak on the scale. The published head still scores each level; its expected level becomes the position `s`, a small new layer gives the spread `σ`, and the probability of level k is the share of a logistic curve around `s` between k − 0.5 and k + 0.5. The format's documentation reports a score answer as the expected level on one dimension, which this matches. True/false and choice keep today's head. The loss is unchanged.
- **`LEVEL_POSITIONS`:** each level is shown with its position, "level 1 of 4 (lowest): …", the same on our data and the benchmark.

The training log shows the loss per question type every 100 steps.

Nothing is uploaded. One T4 is enough.

In [1]:
# Cell 1 — Setup: packages, settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, math, os, random, shutil, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch loads)
from collections import Counter
from pathlib import Path

import pandas as pd
import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

D_SET = "v1_3"                        # data/signpost_d/<D_SET>/systemone.jsonl (v1_3: v1_2 + 299 score cases, L1-L6; v1_4: v1_3 + twin cases, T*, training only)
TRAIN_MODE = "all"                    # "choice": choice only (08b); "all_subsampled": all types cut to 08b's count; "all": every question
ORDINAL_HEAD = False                  # score questions: one-peaked probabilities around a position s with a spread (see the notes)
LEVEL_POSITIONS = True                # score questions: each level shown as "level i of n (lowest/highest): …" (run 15: best)
ORD_LR = 1e-3                         # learning rate of the spread layer (new, so higher than HEAD_LR)
RUN_TAG = TRAIN_MODE + ("-ord" if ORDINAL_HEAD else "") + ("-pos" if LEVEL_POSITIONS else "")   # names the outputs
CHOICE_ONLY = TRAIN_MODE == "choice"  # train, check collapse, pick the epoch and fit the temperature on choice questions only
START = "skamalj/signpost-one-pass"  # the weights we start from (encoder/ + head.pt); "answerdotai/ModernBERT-large" = plain, new head
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.1  # run 7: r 64 (run 6: 16), same alpha / r
HEAD_EXTRA = 2                        # residual blocks added to the head (0 = Signpost's head as published)
LR, HEAD_LR = 2e-4, 5e-4             # as 07 from Signpost; use HEAD_LR 1e-3 for a new head (03, 05)
EPOCHS = 3                            # the best epoch on the dev cases is kept (best at 2 in runs 7, 11 and 12)
BATCH = 8                             # questions per step, one encoder pass each (with gradient checkpointing)
MAX_LEN = 1536                        # state + question + every option in one input (ours ~600 tokens at most; as 05, room for 50 options)
MAX_OPTIONS = 50                      # the grounded-question test independent 2 has up to 50 options (System One allows 26 per choice)
SCORE_DIST_W = 0.5                    # score questions: weight of |expected level - target expected level| / (levels - 1)
LOSS_VARIANTS = [0.0]                 # Brier weights to run, one after another (0.0 = cross-entropy only, as run 7 and 08b variant 1)
BRIER_W = LOSS_VARIANTS[0]            # the weight in use; set by the loop in Cell 6
DEV_FRACTION = 0.15
SEED = 1042                             # the first seed; a collapsed start retries with SEED + 1000, + 2000
COLLAPSE_STEP, COLLAPSE_MARGIN, ATTEMPTS = 400, 0.02, 3   # at this step, dev KL must be this far below a uniform guess's
BENCH = "LocalLLaMA/typed-decisions"  # scored once at the end; never used for training or selection
SEP, ROUTE_Q = " [SEP] ", "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path(f"signpost_d_{RUN_TAG}_run.log")
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()
REV = {START: api.model_info(START).sha, BENCH: api.dataset_info(BENCH).sha}


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | {DEVICE} | "
    f"start {START} @ {REV[START][:8]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 9.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 9.3 MB/s eta 0:00:00:00:0100:01
[05:59:17] torch 2.11.0+cu128 | transformers 5.17.0 | peft 0.21.2 | cuda:0 | start skamalj/signpost-one-pass @ 0cb2271d


In [2]:
# Cell 2a — The Signpost-D cases (System One form): one item per question, with the labellers' averaged distribution as the target

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():                  # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def render(q):
    """A System One question -> ({candidate name: description}, the gold's keys in the same order). One rendering for our
    cases and the benchmark: noul as true/false with their descriptions, choice by its labels, score by its criteria
    strings, lowest first (keys "0".."n-1")."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    assert len(opts) == len(keys), q
    return opts, keys


d_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / D_SET / "systemone.jsonl")]
def dev_split(ids):
    ids = sorted(ids)
    random.Random(SEED).shuffle(ids)
    return set(ids[: int(len(ids) * DEV_FRACTION)])


ADDED = ("I", "K", "L")                   # cases added after v1 (I: invoices, K: choice, L: score cases): each group split on its own,
TRAIN_ONLY = ("T",)                       # twin cases (v1_4): always training, never dev (a twin and its base must not straddle)
dev_ids = dev_split(c["id"] for c in d_cases if not c["id"].startswith(ADDED + TRAIN_ONLY))   # earlier dev cases stay as before
for g in ADDED:
    dev_ids |= dev_split(c["id"] for c in d_cases if c["id"].startswith(g))


def d_items(c):
    out = []
    for qid, q in c["questions"].items():
        opts, keys = render(q)
        gold = c["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "id": c["id"], "qid": qid, "domain": c["meta"]["domain"],
                    "messages": [{"role": "user", "content": state_text(c["state"])}], "current": None,
                    "question": q["instructions"], "options": opts, "levels": list(opts) if q["type"] == "score" else None,
                    "soft": {n: gold[k] for n, k in zip(opts, keys)}})
    return out


train_d = [it for c in d_cases if c["id"] not in dev_ids for it in d_items(c)]
dev_d = [it for c in d_cases if c["id"] in dev_ids for it in d_items(c)]
N_CHOICE_TRAIN = sum(it["task"] == "d_choice" for it in train_d)   # 08b's training set size
if TRAIN_MODE == "choice":
    train_d = [it for it in train_d if it["task"] == "d_choice"]
    dev_d = [it for it in dev_d if it["task"] == "d_choice"]
elif TRAIN_MODE == "all_subsampled":                                 # same count as 08b, all three types
    train_d = random.Random(SEED).sample(train_d, N_CHOICE_TRAIN)
else:
    assert TRAIN_MODE == "all", TRAIN_MODE
print(f"TRAIN_MODE {TRAIN_MODE}: {len(train_d)} training questions (08b had {N_CHOICE_TRAIN}), "
      f"{len({it['id'] for it in train_d})} distinct cases")
print(f"Signpost-D ({D_SET}): {len(d_cases)} cases | train {len(train_d)} questions | dev {len(dev_d)} questions "
      f"({len(dev_ids)} cases) | {dict(Counter(it['task'] for it in train_d))}")

Cloning into 'llms-the-hard-way'...


TRAIN_MODE all: 8249 training questions (08b had 2865), 1786 distinct cases
Signpost-D (v1_3): 2099 cases | train 8249 questions | dev 1438 questions (313 cases) | {'d_noul': 2361, 'd_choice': 2865, 'd_score': 3023}


In [3]:
# Cell 2b — Grounded-question tests (from Signpost): the general reading check

load = lambda p: json.load(open(DATA_DIR / "modernbert_router" / p))
e12 = load("choice_test/items.json")["items"]
ind_q, ind2_q = load("independent/questions.json")["items"], load("independent2/questions.json")["items"]
print(f"E12 {len(e12)} | independent questions {len(ind_q)} | independent 2 questions {len(ind2_q)} "
      f"(up to {max(len(q['options']) for q in ind2_q)} options)")

E12 170 | independent questions 300 | independent 2 questions 200 (up to 50 options)


In [4]:
# Cell 3 — The model: Signpost one-pass's encoder (START) + a new LoRA, one pass per question, each option pooled over its own
# tokens, the published scoring head with HEAD_EXTRA residual blocks (05's OnePassModel; input format as signpost/signpost_one_pass.py)

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer

FROM_SIGNPOST = START != "answerdotai/ModernBERT-large"
START_DIR = Path(snapshot_download(START, revision=REV[START], allow_patterns=None if FROM_SIGNPOST else ["*.json", "*.safetensors", "*.txt"]))
ENC_DIR = START_DIR / "encoder" if FROM_SIGNPOST else START_DIR


class ScoreHead(nn.Module):
    """Signpost's scoring head (Dropout, Linear(H, 256), GELU, Dropout, Linear(256, 1)) with `extra` residual blocks of 256
    units between its two layers. Each block's last layer starts at zero, so a head loaded from Signpost starts out exactly as
    published, and the blocks then learn on top of it."""

    def __init__(self, hidden, extra=0):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, 256), nn.GELU())
        self.blocks = nn.ModuleList([nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 256), nn.GELU(), nn.Linear(256, 256))
                                     for _ in range(extra)])
        for b in self.blocks:
            nn.init.zeros_(b[3].weight); nn.init.zeros_(b[3].bias)
        self.out = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))

    def forward(self, x):
        h = self.inp(x)
        for b in self.blocks:
            h = h + b(h)
        return self.out(h)

    def load_published(self, sd):
        """Signpost's head (a plain Sequential: layers 1 and 4) into this head."""
        self.inp[1].load_state_dict({"weight": sd["1.weight"], "bias": sd["1.bias"]})
        self.out[1].load_state_dict({"weight": sd["4.weight"], "bias": sd["4.bias"]})


class Model:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(ENC_DIR)
        self.tok.padding_side = "right"
        enc = AutoModel.from_pretrained(ENC_DIR, dtype=torch.float32).to(DEVICE)
        enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})   # recompute, don't store, activations
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=["Wqkv", "Wo", "Wi"]))
        self.head = ScoreHead(self.H, HEAD_EXTRA).to(DEVICE)
        self.sig = nn.Linear(self.H, 1).to(DEVICE)          # ORDINAL_HEAD: the spread of a score question's answer
        nn.init.zeros_(self.sig.weight); nn.init.constant_(self.sig.bias, 0.4)   # softplus(0.4) + 0.1 = 1.0 level at the start
        if FROM_SIGNPOST:
            self.head.load_published(torch.load(START_DIR / "head.pt", map_location="cpu")["state_dict"])   # one-pass's head

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode); self.sig.train(mode)

    @staticmethod
    def build(item):
        """messages SEP current agent SEP question SEP option: A SEP option: B ... -> (text, character span of each option)."""
        text = SEP.join([f"{m['role']}: {m['content']}" for m in item["messages"]]
                        + [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"])
        spans = []
        n = len(item["options"])
        for i, (name, desc) in enumerate(item["options"].items()):
            seg = f"{name}. {desc}" if desc else name
            if LEVEL_POSITIONS and item.get("task") == "d_score":    # levels keep their order, lowest first
                seg = f"level {i + 1} of {n}" + (" (lowest)" if i == 0 else " (highest)" if i == n - 1 else "") + ": " + seg
            text += SEP + "option: "
            spans.append((len(text), len(text) + len(seg)))
            text += seg
        return text, spans

    def fit(self, item):
        """The whole input must fit MAX_LEN: a case has one message, so nothing can be dropped (as the loader)."""
        assert len(item["options"]) <= MAX_OPTIONS, f"{len(item['options'])} options > MAX_OPTIONS={MAX_OPTIONS}"
        text, spans = self.build(item)
        n = len(self.tok(text)["input_ids"])
        assert n <= MAX_LEN, f"input of {n} tokens > MAX_LEN={MAX_LEN}"
        return text, spans

    def scores(self, items):
        """-> [items, most options] scores (missing options -1e4): one encoder pass per question, each option the mean of its
        own tokens."""
        built = [self.fit(it) for it in items]
        enc = self.tok([t for t, _ in built], padding=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")                                            # [B, T, 2] characters of each token
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).last_hidden_state
        K = max(len(sp) for _, sp in built)
        spans = torch.tensor([sp + [(0, 0)] * (K - len(sp)) for _, sp in built])     # [B, K, 2]; missing options are empty
        s, e = offsets[..., 0], offsets[..., 1]                                        # [B, T]
        real = enc["attention_mask"].bool() & (e > s)                                  # real, non-special tokens
        member = (real[:, None] & (s[:, None] < spans[..., 1:]) & (e[:, None] > spans[..., :1])).float().to(DEVICE)   # [B, K, T]
        counts = member.sum(-1)                                                        # [B, K] tokens per option
        assert int((counts > 0).sum()) == sum(len(sp) for _, sp in built), "an option got no tokens"
        pooled = (member @ hidden.float()) / counts.clamp(min=1).unsqueeze(-1)         # [B, K, H] mean of each option's tokens
        out = self.head(pooled).squeeze(-1).masked_fill(counts == 0, -1e4)
        if not ORDINAL_HEAD:
            return out
        rows = []
        for i, it in enumerate(items):                       # score questions: one peak around s, spread sigma
            if it.get("task") != "d_score":
                rows.append(out[i])
                continue
            n = len(it["options"])
            k = torch.arange(n, device=DEVICE, dtype=torch.float32)
            s = (torch.softmax(out[i, :n].float(), 0) * k).sum()                       # the published head's expected level
            sigma = F.softplus(self.sig(pooled[i, :n].mean(0))).squeeze() + 0.1      # in levels
            cdf = torch.sigmoid((k[:-1] + 0.5 - s) / sigma)                          # P(level <= k), k = 0 .. n-2
            p = torch.diff(torch.cat([cdf.new_zeros(1), cdf, cdf.new_ones(1)])).clamp(min=1e-6)
            rows.append(torch.cat([torch.log(p / p.sum()), out.new_full((out.shape[1] - n,), -1e4)]))
        return torch.stack(rows)

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
# Cell 4a — The loss: soft labels (+ an expected-level term for score questions, + BRIER_W x Brier); metrics on the dev cases
# (accuracy against the label's top option, KL, Brier, ECE)

def soft_loss(scores, items):
    """Cross-entropy against the labellers' distribution (= KL + a constant). Score questions add
    SCORE_DIST_W x |expected level - target expected level| / (levels - 1), each option weighted by its level in the
    rubric (it["levels"]), not by its place after the options are shuffled."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        lvl = torch.tensor([[it["levels"].index(n) if it["levels"] else 0 for n in it["options"]] + [0] * (K - len(it["options"]))
                            for it in items], device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * lvl).sum(1) - (target * lvl).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    if BRIER_W:                                      # padded options have p ~ 0 and target 0, so they add nothing
        loss = loss + BRIER_W * ((logp.exp() - target) ** 2).sum(1).mean()
    return loss


def d_predictions(model, items, batch=16):
    """-> [(item, {option: probability})] for Signpost-D items."""
    model.train(False)
    out = []
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            p = torch.softmax(model.scores(chunk), dim=1).float().cpu()
            out += [(it, {n: p[i, j].item() for j, n in enumerate(it["options"])}) for i, it in enumerate(chunk)]
    return out


def soften(p, temperature):
    """Probabilities softened as p ** (1 / T), renormalised."""
    q = [max(v, 1e-12) ** (1 / temperature) for v in p]
    z = sum(q)
    return [v / z for v in q]


def t_of(temperature, task):
    """A temperature: one number, or one per question type ({"d_noul": T, ...})."""
    return temperature.get(task, 1.0) if isinstance(temperature, dict) else temperature


def d_metrics(preds, temperature=1.0):
    """The benchmark's four measures, against our own soft labels: accuracy (top option = the label's top option), KL from the
    label, Brier, ECE (10 bins). With a temperature (one, or one per question type), probabilities are softened."""
    rows = []
    for it, p in preds:
        q = {n: max(v, 1e-12) ** (1 / t_of(temperature, it["task"])) for n, v in p.items()}
        z = sum(q.values())
        q = {n: v / z for n, v in q.items()}
        g = it["soft"]
        ok = max(q, key=q.get) == max(g, key=g.get)
        rows.append({"task": it["task"], "ok": ok, "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    return {"all": summ(rows), **{t: summ([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperatures(preds):
    """One temperature per question type, each minimising KL on our own dev cases of that type (never the benchmark)."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    out = {}
    for task in sorted({it["task"] for it, _ in preds}):
        sub = [(it, p) for it, p in preds if it["task"] == task]
        out[task] = min(grid, key=lambda t: d_metrics(sub, t)["all"]["kl"])
    return out


def calibration(pairs):
    """pairs: [(predicted probabilities, gold probabilities)], same option order. Do the predictions sit nearer 0 and 1 than
    the gold? Top-probability histograms side by side, mean normalised entropy, and the over-certain options: predicted
    under 0.01 where the gold has 0.05 or more, with the share of the total KL they cause (scorer's floor 1e-9)."""
    edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0001]
    hist = lambda tops: [sum(lo <= t < hi for t in tops) for lo, hi in zip(edges, edges[1:])]
    ent = lambda v: -sum(x * math.log(x) for x in v if x > 0) / math.log(len(v)) if len(v) > 1 else 0.0
    n_opt = over = 0
    kl_all = kl_over = 0.0
    for p, g in pairs:
        for pi, gi in zip(p, g):
            n_opt += 1
            term = gi * math.log(gi / max(pi, 1e-9)) if gi > 0 else 0.0
            kl_all += term
            if pi < 0.01 and gi >= 0.05:
                over += 1
                kl_over += term
    return {"top_p_bins": [f"{lo:.2f}-{min(hi, 1):.2f}" for lo, hi in zip(edges, edges[1:])],
            "pred_top_hist": hist([max(p) for p, _ in pairs]), "gold_top_hist": hist([max(g) for _, g in pairs]),
            "pred_entropy": round(statistics.mean(ent(p) for p, _ in pairs), 4),
            "gold_entropy": round(statistics.mean(ent(g) for _, g in pairs), 4),
            "over_certain_options": over, "of_options": n_opt,
            "kl_share_from_over_certain": round(kl_over / kl_all, 4) if kl_all else 0.0}

In [6]:
# Cell 4b — The training loop: shuffled batches of Signpost-D questions; after each epoch the dev cases are scored and the
# epoch with the lowest dev KL is kept. Collapse guard: from plain ModernBERT-large a start can stay at chance (Signpost's
# run 22; run 3 here), so at COLLAPSE_STEP the dev KL on 300 questions must beat a uniform guess by COLLAPSE_MARGIN.

class Collapsed(Exception):
    """The predictions stayed at a uniform guess; retry with another seed."""


def uniform_kl(items):
    """KL of a uniform guess from the labels: the level a collapsed model sits at."""
    return statistics.mean(sum(g * math.log(g * len(it["soft"])) for g in it["soft"].values() if g > 0) for it in items)


def batches_for_epoch(epoch, seed):
    d = train_d[:]
    random.Random(seed + epoch).shuffle(d)
    return [d[i:i + BATCH] for i in range(0, len(d), BATCH)]


def train_model(model, seed):
    per_epoch = len(batches_for_epoch(1, seed))
    check = dev_d[:300]
    params = model.lora_params() + list(model.head.parameters()) + (list(model.sig.parameters()) if ORDINAL_HEAD else [])
    groups = [{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}]
    if ORDINAL_HEAD:
        groups.append({"params": model.sig.parameters(), "lr": ORD_LR})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    by_type = {"d_noul": [], "d_choice": [], "d_score": []}             # per-type cross-entropy, for the log only
    total = EPOCHS * per_epoch
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0, history, run = (float("inf"), None, 0), 0, time.perf_counter(), [], []
    log(f"training: {per_epoch} batches per epoch, {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        for items in batches_for_epoch(epoch, seed):
            items = [dict(it) for it in items]
            for it in items:                                    # choice and noul options shuffled; score levels keep their order
                if it["task"] != "d_score":
                    names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            sc = model.scores(items)
            loss = soft_loss(sc, items)
            with torch.no_grad():
                lp = F.log_softmax(sc.float(), dim=1)
                for j, it in enumerate(items):
                    by_type[it["task"]].append(-sum(it["soft"][n] * lp[j, m].item() for m, n in enumerate(it["options"])))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss {statistics.mean(run[-100:]):.4f} | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()} | per type " +
                    " ".join(f"{t[2:]} {statistics.mean(v[-300:]):.3f}" for t, v in by_type.items() if v))
            if step == COLLAPSE_STEP:
                kl, base = d_metrics(d_predictions(model, check))["all"]["kl"], uniform_kl(check)
                log(f"  collapse check at step {step}: dev KL {kl:.4f} vs uniform {base:.4f}")
                if kl > base - COLLAPSE_MARGIN:
                    raise Collapsed(f"dev KL {kl:.4f} at step {step}, uniform guess {base:.4f}")
                model.train(True)
        dev = d_metrics(d_predictions(model, dev_d))["all"]
        history.append({"epoch": epoch, **dev})
        log(f"epoch {epoch} done | dev {dev}")
        if dev["kl"] < best[0]:
            best = (dev["kl"], {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                "head": copy.deepcopy(model.head.state_dict()), "sig": copy.deepcopy(model.sig.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    model.sig.load_state_dict(best[1]["sig"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (dev KL {best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": best[2], "history": history}

In [7]:
# Cell 5a — Timing and the question test (from Signpost's 03)

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": [{"role": "user", "content": it["state"]}], "current": None,
                                    "question": it["question"], "options": {o: "" for o in it["options"]},
                                    "task": {"noul": "d_noul", "score": "d_score"}.get(it["type"], "d_choice")})
        top = int(p.argmax())
        rows.append({"type": it["type"], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [8]:
# Cell 5b — The grounded-question tests on a model: R = {test: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}


def evaluate(model):
    model.train(False)
    R, LAT = {}, {}
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        rows, LAT[name] = questions(model, items)
        R[f"{name} questions"] = (sum(r["ok"] for r in rows), len(rows))
        for t in TYPE:
            rs = [r for r in rows if r["type"] == t]
            if rs:
                R[f"{name} {TYPE[t]}"] = (sum(r["ok"] for r in rs), len(rs))
        log(f"{name} questions {R[f'{name} questions'][0]}/{len(rows)} | " +
            " · ".join(f"{TYPE[t]} {R[f'{name} {TYPE[t]}'][0]}/{R[f'{name} {TYPE[t]}'][1]}" for t in TYPE if f"{name} {TYPE[t]}" in R))
    return R, LAT

In [9]:
# Cell 5c — The benchmark, scored once at the end (general mode: never used for training or for any choice above).
# Scoring copied from the benchmark's bench/score_with_jobs.py (score_one, 10-bin ECE).

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":
        pred = dict(pred)
        pred["probabilities"] = {{"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()}
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p, am_g = p.index(max(p)), g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def bench_item(case, q):
    """A benchmark question as a Signpost item, through the same render() as our own cases (state as its JSON text, as
    /v1/systemone sends it)."""
    opts, keys = render(q)
    return {"messages": [{"role": "user", "content": state_text(case["state"])}], "current": None,
            "question": q.get("instructions") or "", "options": opts}, keys


def bench_score(model, cases, temperature=1.0, pairs=None):
    """The four measures on the benchmark. temperature: one number, or one per question type. pairs (a list), if given,
    collects (prediction, gold) in gold order for calibration()."""
    model.train(False)
    rows = []
    for case in cases:
        for qid, q in case["questions"].items():
            item, keys = bench_item(case, q)
            p = soften(model.probs(item).tolist(), t_of(temperature, "d_" + q["type"]))
            pred = {"probabilities": dict(zip(keys, p))}
            ok, kl, brier, conf = score_one(pred, case["gold"][qid])
            rows.append({"type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
            if pairs is not None:
                g = case["gold"][qid]["probabilities"]
                pairs.append(([pred["probabilities"].get(k, 0.0) for k in g], [float(v) for v in g.values()]))
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    out = {"all": summ(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    return out

In [10]:
# Cell 5d — Benchmark items carry their documented type, so a score question gets the score head and level positions

_bench_item = bench_item


def bench_item(case, q):
    item, keys = _bench_item(case, q)
    item["task"] = "d_" + q["type"]
    return item, keys

In [11]:
# Cell 6 — For each Brier weight in LOSS_VARIANTS: train (same start and seed), measure, save; then free the GPU

SUMMARY = {}
for BRIER_W in LOSS_VARIANTS:
    log(f"===== variant: BRIER_W = {BRIER_W} ===== | {gpu()}")

    if BRIER_W == LOSS_VARIANTS[0] and AMP and torch.cuda.memory_allocated(DEVICE) > 1e9:
        raise RuntimeError("the GPU already holds an earlier model: restart the kernel (or start a new session) and run all cells "
                           "once, from the top")
    attempts = []
    for attempt in range(ATTEMPTS):
        seed = SEED + 1000 * attempt
        random.seed(seed); torch.manual_seed(seed)
        model = Model()
        if attempt == 0:
            model.enc.print_trainable_parameters()
        log(f"attempt {attempt + 1} (seed {seed})")
        try:
            trained = train_model(model, seed)
            attempts.append({"seed": seed, "status": "ok"})
            break
        except Collapsed as e:
            log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
            attempts.append({"seed": seed, "status": f"collapsed: {e}"})
            model = None; gc.collect()
            if AMP:
                torch.cuda.empty_cache()
    else:
        raise RuntimeError(f"training collapsed in every attempt: {attempts}")

    dev_preds = d_predictions(model, dev_d)
    T = fit_temperatures(dev_preds)                            # one per question type, on our dev cases
    AFTER = {"T=1": d_metrics(dev_preds), "T per type": d_metrics(dev_preds, T)}
    for k, v in AFTER.items():
        log(f"after training, dev cases, {k}: " + " | ".join(f"{t} {m}" for t, m in v.items()))
    for name, part in (("v1 cases (as run 6)", [(it, p) for it, p in dev_preds if not it["id"].startswith(ADDED)]),
                       ("added cases (invoices)", [(it, p) for it, p in dev_preds if it["id"].startswith("I")]),
                       ("added cases (choice, K)", [(it, p) for it, p in dev_preds if it["id"].startswith("K")]),
                       ("added cases (score, L)", [(it, p) for it, p in dev_preds if it["id"].startswith("L")])):
        if part:
            log(f"after training, dev {name}, T per type: {d_metrics(part, T)['all']}")

    R, LAT = evaluate(model)                                   # grounded-question tests after training (compare Signpost in the ledger)
    log(f"temperatures (dev): {T}")
    TEST = bench_split("test")
    pairs_t1, pairs_t = [], []
    BENCH_RES = {"T=1": bench_score(model, TEST, 1.0, pairs_t1), "T per type": bench_score(model, TEST, T, pairs_t)}
    CALIB = {"dev, T=1": calibration([([p[n] for n in it["options"]], [it["soft"][n] for n in it["options"]]) for it, p in dev_preds]),
             "benchmark, T=1": calibration(pairs_t1), "benchmark, T per type": calibration(pairs_t)}
    for k, v in CALIB.items():
        log(f"calibration, {k}: {v}")
    for k, v in BENCH_RES.items():
        log(f"benchmark test, {k}: {v['all']}")
        for g, m in v.items():
            if g != "all":
                print(f"  {g:38s} acc {m['accuracy']:.3f}  KL {m['kl_from_gold']:.3f}  Brier {m['brier']:.3f}  ECE {m['ece']:.3f}  (n {m['n']})")
    print("reference, benchmark choice: run 7 (08, all types) acc 0.430 KL 0.601 Brier 0.281 ECE 0.140 · run 9 (09 decoder) acc 0.452\n"
          "reference, our choice dev: run 7 acc 0.690 KL 0.488 (v1_1 dev) · overall benchmark: run 7 0.471, prior 0.470")
    json.dump({"d_set": D_SET, "run_tag": RUN_TAG, "train_mode": TRAIN_MODE, "ordinal_head": ORDINAL_HEAD, "level_positions": LEVEL_POSITIONS, "n_train": len(train_d), "brier_w": BRIER_W, "revisions": REV, **trained, "attempts": attempts, "dev_after": AFTER, "temperature": T, "calibration": CALIB,
               "question_tests": {k: list(v) for k, v in R.items()}, "latency": {k: lat_summary(v) for k, v in LAT.items()},
               "benchmark": BENCH_RES}, open(f"signpost_d_{RUN_TAG}_results_{D_SET}_brier{BRIER_W:g}.json", "w"), indent=1)


    OUT = Path(f"signpost-d-{RUN_TAG}-{D_SET}-brier{BRIER_W:g}")
    shutil.rmtree(OUT, ignore_errors=True)
    model.enc.save_pretrained(OUT / "adapter")
    merged = model.enc.merge_and_unload()
    merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
    model.tok.save_pretrained(OUT / "encoder")
    torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H,
                "extra_blocks": HEAD_EXTRA, "ordinal_sigma": ({k: v.float().cpu() for k, v in model.sig.state_dict().items()}
                                                          if ORDINAL_HEAD else None)}, OUT / "head.pt")
    json.dump({"name": f"Signpost-D one-pass ({RUN_TAG})", "train_mode": TRAIN_MODE, "ordinal_head": ORDINAL_HEAD, "level_positions": LEVEL_POSITIONS, "format": "one-pass", "start": START, "start_revision": REV[START], "sep": SEP,
               "max_len": MAX_LEN, "max_options": MAX_OPTIONS, "route_question": ROUTE_Q, "temperature": T, "d_set": D_SET,
               "best_epoch": trained["best_epoch"], "seed_attempts": attempts, "lora_r": LORA_R, "head_extra_blocks": HEAD_EXTRA, "lora_alpha": LORA_ALPHA},
              open(OUT / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent / "signpost" / "signpost_one_pass.py", OUT / "signpost_one_pass.py")
    shutil.make_archive(str(OUT), "zip", root_dir=".", base_dir=str(OUT))
    log(f"saved {OUT}/ and {OUT}.zip ({Path(str(OUT) + '.zip').stat().st_size / 1e9:.2f} GB) | download it if the result is worth keeping")
    b = BENCH_RES["T per type"]
    SUMMARY[f"{RUN_TAG}, BRIER_W {BRIER_W:g}"] = {"best epoch": trained["best_epoch"], "dev choice": AFTER["T per type"]["d_choice"],
                                     "dev score": AFTER["T per type"]["d_score"], "bench choice": b.get("type=choice"),
                                     "bench score": b.get("type=score"), "bench noul": b.get("type=noul"), "bench all": b["all"]}
    model = merged = None; gc.collect()
    if AMP:
        torch.cuda.empty_cache()

print("\nbenchmark (T per type) | references, acc / KL: run 7 all 0.471 / 0.548, choice 0.430 / 0.601, score 0.384, true/false 0.628 · "
      "run 12 (08c) all 0.441 / 0.510, choice 0.353 / 0.482, score 0.362 / 0.695, true/false 0.633 / 0.292")
for k, v in SUMMARY.items():
    c = v["bench choice"]
    log(f"{k}: best epoch {v['best epoch']} | dev choice acc {v['dev choice']['accuracy']} KL {v['dev choice']['kl']} | "
        f"bench choice acc {c['accuracy']} KL {c['kl_from_gold']} | score acc {v['bench score']['accuracy']} KL {v['bench score']['kl_from_gold']} | "
        f"true/false acc {v['bench noul']['accuracy']} KL {v['bench noul']['kl_from_gold']} | all acc {v['bench all']['accuracy']} KL {v['bench all']['kl_from_gold']}")
json.dump(SUMMARY, open(f"signpost_d_{RUN_TAG}_summary_{D_SET}.json", "w"), indent=1)

[05:59:33] ===== variant: BRIER_W = 0.0 ===== | GPU 0.0 GB (peak 0.0)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

trainable params: 28,786,688 || all params: 423,568,384 || trainable%: 6.7962
[05:59:36] attempt 1 (seed 1042)
[05:59:36] training: 1032 batches per epoch, 3096 steps


/tmp/ipykernel_114/2598679269.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[06:01:33]   epoch 1 step 100/3096 | loss 1.2932 | ~58.5 min left | GPU 2.1 GB (peak 3.0) | per type noul 0.851 choice 1.311 score 1.303
[06:03:45]   epoch 1 step 200/3096 | loss 1.1558 | ~60.1 min left | GPU 2.1 GB (peak 3.0) | per type noul 0.690 choice 1.089 score 1.237
[06:05:54]   epoch 1 step 300/3096 | loss 1.0300 | ~58.8 min left | GPU 2.1 GB (peak 3.1) | per type noul 0.600 choice 1.082 score 1.100
[06:08:04]   epoch 1 step 400/3096 | loss 1.0604 | ~57.0 min left | GPU 2.1 GB (peak 3.1) | per type noul 0.613 choice 0.999 score 1.165
[06:08:18]   collapse check at step 400: dev KL 0.4634 vs uniform 0.6641
[06:10:29]   epoch 1 step 500/3096 | loss 1.0305 | ~56.5 min left | GPU 2.1 GB (peak 3.1) | per type noul 0.606 choice 0.955 score 1.147
[06:12:42]   epoch 1 step 600/3096 | loss 0.9978 | ~54.5 min left | GPU 2.1 GB (peak 3.1) | per type noul 0.543 choice 0.970 score 1.117
[06:14:54]   epoch 1 step 700/3096 | loss 1.0081 | ~52.4 min left | GPU 2.1 GB (peak 3.1) | per type noul

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

: 

## Reading the results

- **Benchmark overall and by type** against run 7 (0.471 / KL 0.548; choice 0.430, score 0.384, true/false 0.628) and run 12 (0.441 / KL 0.510).
- **With `ORDINAL_HEAD` or `LEVEL_POSITIONS`:** compare score questions (benchmark and dev) with the run without them; the other types should not move.
- **Per-type loss in the log:** whether score questions train as easily as the others.
- **Dev parts:** v1 (as in every run since run 6), invoices, K and L cases.


In [ ]:
# Dev predictions for failure analysis (our own dev cases; reading only)
import json
DEV_DUMP = [{"id": it["id"], "qid": it["qid"], "t": it["task"][2:],
             "p": [round(p[n], 4) for n in it["options"]]} for it, p in dev_preds]
json.dump(DEV_DUMP, open(f"dev_preds_{RUN_TAG}.json", "w"))
print("DEVPRED_JSON " + json.dumps(DEV_DUMP, separators=(",", ":")))
print(f"{len(DEV_DUMP)} dev questions dumped")
